# HW03-ICA :: Part A — Input Pipeline (Horses vs. Camels)

COSC 6373

Adam Nelson-Archer, 2140122


## Dataset setup

Download the Kaggle dataset ("Horses and Camels"), unzip it, and place it somewhere accessible (e.g., OneDrive).

This notebook expects a directory that contains **exactly two class folders** (names can vary):

```
some_folder/
  horses/
    ... .jpg/.png ...
  camels/
    ... .jpg/.png ...
```

If your dataset is nested (e.g., `train/` and `test/`), you can point `DATA_DIR` to the split you want to parse (e.g., the `train` folder).


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import random

import matplotlib.pyplot as plt
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Working directory:", Path.cwd())


In [ ]:
# -----------------------------
# Configure where your dataset lives
# -----------------------------
# Option A: point directly to the dataset folder you unzipped
# DATA_DIR = Path(r"C:\\Users\\Adam\\OneDrive\\...\\horses-and-camels")

# Option B: keep it near the repo (if you copied it here)
# DATA_DIR = Path("./data/horses_and_camels")

# For grading: set this to the correct path on your machine.
DATA_DIR = Path(r"C:\\path\\to\\horses_and_camels_dataset")

# Image pipeline settings
IMG_SIZE = (224, 224)   # ResNet50 default input size
BATCH_SIZE = 32
SEED = 6373

print("DATA_DIR =", DATA_DIR)


In [ ]:
def _is_image_file(p: Path) -> bool:
    return p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"}


def describe_folder(root: Path, max_files: int = 5) -> None:
    """Print a short, human-readable description of a dataset folder."""
    print("\nFolder:", root)
    if not root.exists():
        print("  - does not exist")
        return
    if not root.is_dir():
        print("  - not a directory")
        return

    class_dirs = [d for d in root.iterdir() if d.is_dir() and not d.name.startswith(".")]
    print("  - immediate subfolders:", [d.name for d in class_dirs])
    for d in class_dirs:
        imgs = [p for p in d.iterdir() if p.is_file() and _is_image_file(p)]
        print(f"  - {d.name}: {len(imgs)} image(s)")
        for p in imgs[:max_files]:
            print("      ", p.name)


def find_class_folder(root: Path) -> Path:
    """Return a folder that looks like: root/<class_a>/..., root/<class_b>/...

    If `root` already matches this structure, returns `root`.
    Otherwise, searches 2 levels deep for a matching folder.
    """
    if not root.exists():
        raise FileNotFoundError(f"DATA_DIR does not exist: {root}")
    if not root.is_dir():
        raise NotADirectoryError(f"DATA_DIR is not a directory: {root}")

    def looks_like_class_dir(p: Path) -> bool:
        subdirs = [d for d in p.iterdir() if d.is_dir() and not d.name.startswith(".")]
        if len(subdirs) < 2:
            return False
        # At least two subfolders should contain images
        img_counts = []
        for d in subdirs:
            img_counts.append(sum(1 for x in d.iterdir() if x.is_file() and _is_image_file(x)))
        return sum(1 for c in img_counts if c > 0) >= 2

    if looks_like_class_dir(root):
        return root

    # Search up to 2 levels deep (common Kaggle zip layouts)
    candidates = []
    for p in root.rglob("*"):
        if p.is_dir() and p != root:
            # Limit depth to avoid expensive scans
            try:
                rel = p.relative_to(root)
            except ValueError:
                continue
            if len(rel.parts) > 2:
                continue
            if looks_like_class_dir(p):
                candidates.append(p)

    if not candidates:
        describe_folder(root)
        raise ValueError(
            "Could not find a dataset folder containing 2+ class subfolders with images.\n"
            "Update DATA_DIR to point to the folder that contains the class folders (e.g., horses/, camels/)."
        )

    # Pick the shallowest candidate
    candidates.sort(key=lambda p: len(p.relative_to(root).parts))
    return candidates[0]


class_root = find_class_folder(DATA_DIR)
print("Using class folder:", class_root)
describe_folder(class_root)


In [ ]:
# -----------------------------
# Build the input pipeline
# -----------------------------
# Keras utility builds a tf.data pipeline from a directory of class subfolders.

train_ds = tf.keras.utils.image_dataset_from_directory(
    class_root,
    labels="inferred",
    label_mode="int",
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    class_root,
    labels="inferred",
    label_mode="int",
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print("Class names:", class_names)


In [ ]:
# Performance optimizations (typical tf.data best practices)
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("train_ds:", train_ds)
print("val_ds:", val_ds)


In [ ]:
# Inspect one batch to confirm tensor shapes
images, labels = next(iter(train_ds))
print("images shape:", images.shape)   # (batch, H, W, C)
print("labels shape:", labels.shape)   # (batch,)
print("dtype:", images.dtype, labels.dtype)


In [ ]:
# Visualize a few examples from the batch
plt.figure(figsize=(10, 10))
for i in range(min(9, images.shape[0])):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(tf.cast(images[i], tf.uint8))
    plt.title(class_names[int(labels[i])])
    plt.axis("off")
plt.suptitle("Sample images from the input pipeline")
plt.show()


## Notes

- `image_dataset_from_directory(...)` returns a `tf.data.Dataset` that yields `(images, labels)` batches.
- The pipeline above includes **caching**, **shuffling** (train only), and **prefetching** for performance.
- For transfer learning with ResNet50 later, you typically add preprocessing:
  - `tf.keras.applications.resnet50.preprocess_input` (normalization) and optional data augmentation.


## Acknowledgment

I used a coding assistant (ChatGPT in Cursor, GPT‑5.2) to help generate and organize parts of this notebook.
